In [0]:
%python
import pytest
from pyspark.sql import functions as F


In [0]:
show tables from credit_analysis_catalog.silver;

In [0]:
%python
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

CATALOG = "credit_analysis_catalog"


# SILVER - APPLICANT PROFILES

def test_silver_applicant_profiles_not_empty():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    assert df.count() > 0, \
        "Silver applicant_profiles table is empty"


def test_silver_applicant_profiles_applicant_id_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    null_count = df.filter(
        "applicant_id IS NULL"
    ).count()

    assert null_count == 0, \
        f"Found {null_count} null applicant_id values"


def test_silver_applicant_profiles_no_duplicate_applicants():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    duplicate_count = (
        df.groupBy("applicant_id")
        .count()
        .filter("count > 1")
        .count()
    )

    assert duplicate_count == 0, \
        f"Found {duplicate_count} duplicate applicant_id groups"


def test_silver_applicant_profiles_valid_region():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    invalid_count = df.filter(
        """
        region IS NULL
        OR region NOT IN (
            'Central',
            'North',
            'North-East',
            'South'
        )
        """
    ).count()

    assert invalid_count == 0, \
        f"Found {invalid_count} invalid region values"


def test_silver_applicant_profiles_valid_gender():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    invalid_count = df.filter(
        """
        gender IS NULL
        OR gender NOT IN (
            'Female',
            'Joint',
            'Male',
            'Sex Not Available'
        )
        """
    ).count()

    assert invalid_count == 0, \
        f"Found {invalid_count} invalid gender values"


def test_silver_applicant_profiles_age_not_null():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    null_count = df.filter(
        "age IS NULL"
    ).count()

    assert null_count == 0, \
        f"Found {null_count} null age values"


def test_silver_applicant_profiles_income_non_negative():

    df = spark.table(
        f"{CATALOG}.silver.silver_applicant_profiles"
    )

    invalid_count = df.filter(
        "income < 0"
    ).count()

    assert invalid_count == 0, \
        f"Found {invalid_count} negative income values"



# RUN APPLICANT PROFILE TESTS


test_functions = [

    test_silver_applicant_profiles_not_empty,
    test_silver_applicant_profiles_applicant_id_not_null,
    test_silver_applicant_profiles_no_duplicate_applicants,
    test_silver_applicant_profiles_valid_region,
    test_silver_applicant_profiles_valid_gender,
    test_silver_applicant_profiles_age_not_null,
    test_silver_applicant_profiles_income_non_negative,

]


# TEST RUNNER

passed = 0
failed = 0

for test in test_functions:

    try:

        test()

        print(f"PASSED: {test.__name__}")

        passed += 1

    except AssertionError as e:

        print(f"FAILED: {test.__name__}")
        print(f"       {e}")

        failed += 1

    except Exception as e:

        print(f"ERROR: {test.__name__}")
        print(f"       {e}")

        failed += 1


print("\n" + "=" * 50)
print("SILVER APPLICANT PROFILES TEST SUMMARY")
print("=" * 50)

print(f"Total Tests : {len(test_functions)}")
print(f"Passed      : {passed}")
print(f"Failed      : {failed}")

success_rate = (
    passed / len(test_functions) * 100
    if len(test_functions) > 0
    else 0
)

print(f"Success %   : {success_rate:.2f}%")